# Ablation Significance Analysis

Checks the three ablation studies (`ablations/ablation1_model_comparison.py`,
`ablation2_curve_preprocessing.py`, `ablation3_generalisability.py`) for statistically
significant differences between conditions, reusing the exact same Friedman + pairwise
Wilcoxon (Holm-Bonferroni corrected) / McNemar framework as `08_statistical_comparison.py`
-- no test logic is reimplemented here, just adapted to the ablation scripts' flat
per-dataset `*_performances.joblib` schema (same per-model keys as
`classification_performances.joblib`, just one dataset's results per file instead of
nested under `{dataset_name: {mode: {...}}}`).

**Primary test (N >= 2 datasets):** Friedman test across datasets (one mean-accuracy
value per dataset per condition -> independent observations), then pairwise Wilcoxon
signed-rank with Holm-Bonferroni correction, effect size = rank-biserial correlation.

**Fallback (N = 1 dataset):** McNemar's test on concatenated per-sample outcomes.


In [ ]:
import os, sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch, FancyArrowPatch
from pathlib import Path

_NB_DIR = Path.cwd()
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        _NB_DIR = Path(_nb).resolve().parent
except Exception:
    pass
_ROOT = _NB_DIR.parent.parent  # main_code/
sys.path.insert(0, str(_ROOT))
sys.path.insert(0, str(_ROOT / "utils"))
sys.path.insert(0, str(_ROOT / "utils" / "model_training"))

%load_ext autoreload
%autoreload 2
import config
import statistical_comparison as stat_comparison

%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'font.size': 10,
})
print("Notebook dir:", _NB_DIR)
print("main_code root:", _ROOT)

In [ ]:
from IPython.display import HTML, display

# Same 5 LAB_DDM_paper folders the user's ablation-1/3 SLURM array jobs target
# (01, 02, 03, 09, 10 -- see abl_ablation1_model_comparison.sh / abl_ablation3_generalisability.sh).
LAB_DATASETS = ['01_ACA_qdPCR', '02_AMCA_qdLAMP', '03_AMCA_qdPCR']  # main_code lab/training.py output

# Mirrors each ablation script's MODELS list exactly -- keep these in sync if the
# scripts' lists change.
ABLATION1_MODELS = ['knn', 'cnn', 'gru', 'transformer', 'cnn_gru_dual', 'cnn_gru_dual_attn_recon']

ALPHA = 0.05
METRIC = 'accuracy'

def show_result(stats, figs, test_type, metric_name='Accuracy', alpha=ALPHA, title=None):
    """Render one comparison's summary table + figures inline, reusing
    stat_comparison's HTML/plot builders untouched."""
    if title:
        display(HTML(f'<h3>{title}</h3>'))
    if not stats or 'error' in (stats or {}):
        display(HTML(f'<p style="color:#e74c3c">{(stats or {}).get("error", "No data.")}</p>'))
        return
    html = stat_comparison.build_tab_content(stats, figs, test_type, metric_name, alpha)
    display(HTML(html))
    for fig in figs.values():
        display(fig)
        plt.close(fig)


In [ ]:
def load_ablation_exp_data(joblib_path, models, curve_type='ori_curve', metric=METRIC):
    """Adapt one ablation *_performances.joblib (flat {filter: {...}} schema --
    same per-model y_preds_AC_*_/y_trues_ keys as classification_performances.joblib,
    just one dataset's results, not nested under {dataset_name: {mode: {...}}}) into
    the {curve_type: {filter: {model: {fold_metrics, mean_metric, y_true_all,
    y_pred_all, n_folds}}}} shape stat_comparison.build_accuracy_matrix/_get_value
    expect. Mirrors stat_comparison.load_exp_data's inner loop exactly."""
    joblib_path = Path(joblib_path)
    if not joblib_path.exists():
        return None
    raw = joblib.load(joblib_path)

    out_filters = {}
    for filter_key, filter_results in raw.items():
        if "y_trues_" not in filter_results:
            continue
        y_trues_ = filter_results["y_trues_"]
        n_folds = len(y_trues_)
        if n_folds == 0:
            continue

        out_filters[filter_key] = {}
        for model_key in models:
            preds_key = config.MODEL_KEY_MAP[model_key][0]
            if preds_key not in filter_results:
                continue
            y_preds_ = filter_results[preds_key]
            if len(y_preds_) != n_folds:
                continue

            fold_metrics, y_true_parts, y_pred_parts = [], [], []
            for fi in range(n_folds):
                y_true = np.asarray(y_trues_[fi])
                y_pred = np.asarray(y_preds_[fi])
                if len(y_true) == 0 or len(y_true) != len(y_pred):
                    continue
                fold_metrics.append(stat_comparison._compute_metric(y_true, y_pred, metric))
                y_true_parts.append(y_true)
                y_pred_parts.append(y_pred)

            if not fold_metrics:
                continue
            out_filters[filter_key][model_key] = {
                "fold_metrics": fold_metrics,
                "mean_metric":  float(np.mean(fold_metrics)),
                "y_true_all":   np.concatenate(y_true_parts),
                "y_pred_all":   np.concatenate(y_pred_parts),
                "n_folds":      len(fold_metrics),
            }
    return {curve_type: out_filters}


def load_ablation_across_datasets(exp_folder, dataset_names, joblib_name, models,
                                  curve_type='ori_curve', metric=METRIC):
    """One (name, exp_data) pair per dataset with usable results -- the shape
    stat_comparison.run_comparison's exp_data_list expects (one entry per
    independent 'experiment' for the Friedman test)."""
    exp_data_list = []
    for name in dataset_names:
        path = Path(exp_folder) / name / "ablations" / joblib_name
        data = load_ablation_exp_data(path, models, curve_type=curve_type, metric=metric)
        if data is not None and any(data[curve_type].values()):
            exp_data_list.append((name, data))
        else:
            print(f'  [WARN] no usable results for {name} at {path}')
    return exp_data_list


## Ablation 1 -- Model Comparison

kNN vs CNN vs GRU vs Transformer vs cnn_gru_dual vs cnn_gru_dual_attn_recon, across the 5 LAB_DDM_paper datasets. `cnn_gru_dual_attn_recon` is soft-skipped on LAB data (no pixel-grid metadata) -- if it has zero results across all 5 datasets it's automatically dropped from the comparison by `build_accuracy_matrix` (conditions with any missing data are excluded).

In [ ]:
exp_data_1 = load_ablation_across_datasets(
    config.LAB_EXP_FOLDER, LAB_DATASETS,
    "model_comparison_performances.joblib", ABLATION1_MODELS)

stats1, figs1, test_type1, _ = stat_comparison.run_comparison(
    exp_data_1,
    compare_axis="models",
    all_conditions=ABLATION1_MODELS,
    condition_print_map=config.MODEL_PRINT_MAP,
    fixed={"curve_type": "ori_curve", "filter": None},
    baseline_key="cnn_gru_dual",
    metric=METRIC,
    alpha=ALPHA,
)
show_result(stats1, figs1, test_type1, title="Ablation 1: Model Comparison")


In [ ]:
config.MODEL_PRINT_MAP

In [ ]:
DATASET_LABELS_1 = {
    '01_ACA_qdPCR':   'qdPCR (3-plex)\n(Moniri et al., 2020a)',
    '03_AMCA_qdPCR':  'qdPCR (9-plex)\n(Moniri et al., 2020b)',
    '02_AMCA_qdLAMP': 'qdLAMP (5-plex)\n(Malpartida-Cardenas, et al., 2022)',
}
DATASET_ORDER_1 = ['01_ACA_qdPCR', '03_AMCA_qdPCR', '02_AMCA_qdLAMP']
BASELINE_MODELS_1 = {'01_ACA_qdPCR': 'knn', '03_AMCA_qdPCR': 'cnn', '02_AMCA_qdLAMP': 'knn'}

MODEL_COLORS_1 = {
    'knn':                     '#85C1E9',
    'cnn':                     '#82E0AA',
    'gru':                     '#F5B041',
    'transformer':             '#EC7063',
    'cnn_gru_dual':            '#BB8FCE',
    'cnn_gru_dual_attn_recon': '#F7DC6F',
}

MODEL_PRINT_MAP = {
    'knn':                     'KNN',
    'cnn':                     'CNN',
    'gru':                     'BiGRU',
    'transformer':             'Transformer',
    'cnn_gru_dual':            'CNN-BiGRU Dual Branch',
    'cnn_gru_dual_attn_recon': 'CNN-BiGRU Dual + Attn Recon',
}

def plot_ablation1_barchart(exp_data, models, baseline_map, model_colors, dataset_order=None,
                            dataset_labels=None, curve_type='ori_curve', filter_key=None):
    data_by_name = dict(exp_data)
    dataset_names = [n for n in (dataset_order or data_by_name) if n in data_by_name]
    present_models = [m for m in models
                       if any(m in data_by_name[n][curve_type].get(filter_key, {}) for n in dataset_names)]

    acc = np.full((len(dataset_names), len(present_models)), np.nan)
    for i, name in enumerate(dataset_names):
        entry = data_by_name[name][curve_type].get(filter_key, {})
        for j, m in enumerate(present_models):
            if m in entry:
                acc[i, j] = entry[m]['mean_metric'] * 100
    best_j = np.nanargmax(acc, axis=1)

    n_models = len(present_models)
    width = 0.8 / n_models
    x = np.arange(len(dataset_names))

    fig, ax = plt.subplots(figsize=(2 * len(dataset_names) + 2, 4))
    for j, m in enumerate(present_models):
        offsets = x + (j - (n_models - 1) / 2) * width
        bars = ax.bar(offsets, acc[:, j], width=width, color=model_colors.get(m, '#95A5A6'),
                      label=MODEL_PRINT_MAP.get(m, m), zorder=3)
        for i, bar in enumerate(bars):
            v = acc[i, j]
            if np.isnan(v):
                continue
            is_baseline = baseline_map.get(dataset_names[i]) == m
            is_best = j == best_j[i]
            if is_baseline:
                bar.set_linestyle('--')
                bar.set_edgecolor('#555555')
                bar.set_linewidth(1.3)
            if is_best:
                ax.text(bar.get_x() + bar.get_width() / 2, v + 2.6, '\u2605',
                        ha='center', va='bottom', fontsize=11, color='#B7950B', zorder=4)
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.6, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if is_best else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels([(dataset_labels or {}).get(n, n) for n in dataset_names])
    # ax.set_xlabel('Dataset')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 112)
    fig.suptitle('Phase 1: Accuracy by Dataset and Model', fontsize=13, y=1.03)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=n_models, frameon=False)
    fig.tight_layout()
    return fig


fig_bar1 = plot_ablation1_barchart(
    exp_data_1, ABLATION1_MODELS, BASELINE_MODELS_1, MODEL_COLORS_1,
    dataset_order=DATASET_ORDER_1, dataset_labels=DATASET_LABELS_1,
)
plt.show()


## Ablation 1 (cont.) -- Full Metrics Table

Accuracy, macro-averaged F1/precision/sensitivity/specificity, MCC, and per-target
sensitivity/specificity, for every (dataset, model) pair -- one table per dataset since
each LAB dataset has a different target panel (3/9/5-plex).

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder


def get_class_names(exp_path):
    exp_path = Path(exp_path)
    training_data = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    Y_well = training_data["Y_well"]
    label_mappings = config.get_label_mappings(exp_path)
    if exp_path.name in label_mappings:
        mapping = label_mappings[exp_path.name]
        Y_well = [mapping.get(w, w) for w in Y_well]
    return LabelEncoder().fit(Y_well).classes_


def compute_full_metrics(y_true, y_pred, class_names):
    labels = np.arange(len(class_names))
    report = classification_report(y_true, y_pred, labels=labels, target_names=list(class_names),
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    row = {
        "n_test":      len(y_true),
        "accuracy":    accuracy_score(y_true, y_pred),
        "precision":   report["macro avg"]["precision"],
        "sensitivity": report["macro avg"]["recall"],
        "f1":          report["macro avg"]["f1-score"],
        "mcc":         stat_comparison._compute_metric(y_true, y_pred, "mcc"),
    }
    specs = []
    for i, cls in enumerate(class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - cm[i, :].sum() - fp
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        specs.append(spec)
        row[f"sensitivity__{cls}"] = report[cls]["recall"]
        row[f"specificity__{cls}"] = spec
    row["specificity"] = float(np.nanmean(specs))
    return row


metrics_rows = []
for dataset_name, exp_data in exp_data_1:
    class_names = get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name)
    entry = exp_data["ori_curve"].get(None, {})
    for model_key in ABLATION1_MODELS:
        if model_key not in entry:
            continue
        d = entry[model_key]
        m = compute_full_metrics(d["y_true_all"], d["y_pred_all"], class_names)
        metrics_rows.append(dict(dataset=dataset_name, model=model_key, **m))

metrics_df = pd.DataFrame(metrics_rows)
print(f"Built metrics for {len(metrics_df)} (dataset, model) pairs.")
metrics_df[["dataset", "model", "n_test", "accuracy", "f1", "sensitivity", "specificity", "mcc"]]

In [ ]:
model_order = [m for m in ABLATION1_MODELS if m in metrics_df.model.unique()]

for dataset_name, _ in exp_data_1:
    class_names = get_class_names(Path(config.LAB_EXP_FOLDER) / dataset_name)
    sub = metrics_df[metrics_df.dataset == dataset_name].set_index("model").reindex(model_order).dropna(how="all")

    data = {
        ("Overall", "N"):                  sub["n_test"],
        ("Overall", "Accuracy"):           sub["accuracy"],
        ("Overall", "F1 (macro)"):         sub["f1"],
        ("Overall", "Precision (macro)"):  sub["precision"],
        ("Overall", "MCC"):                sub["mcc"],
    }
    for c in class_names:
        data[("Sensitivity", c)] = sub[f"sensitivity__{c}"]
    data[("Sensitivity", "Macro")] = sub["sensitivity"]
    for c in class_names:
        data[("Specificity", c)] = sub[f"specificity__{c}"]
    data[("Specificity", "Macro")] = sub["specificity"]

    table = pd.DataFrame(data)
    table.index = [config.MODEL_PRINT_MAP.get(m, m) for m in table.index]

    pct_cols = [c for c in table.columns if c[1] not in ("N", "MCC")]
    best_cols = pct_cols + [("Overall", "MCC")]  # higher-is-better columns; N excluded
    display_table = table.copy()
    display_table[pct_cols] = display_table[pct_cols] * 100

    fmt = {c: "{:.2f}%" for c in pct_cols}
    fmt[("Overall", "N")] = "{:.0f}"
    fmt[("Overall", "MCC")] = "{:.3f}"

    styled = (display_table.style
             .format(fmt)
             .highlight_max(subset=best_cols, color="#a8e6a1")
             .set_caption(f"{dataset_name} -- Ablation 1 full metrics (row = model)"))
    display(styled)

## KNN Intuition -- Curve Shape Similarity (01_ACA_qdPCR)

One representative real curve per target (the real curve closest to that target's mean shape),
plus one held-out random sample labelled "?" -- illustrates that raw curve shape alone already
separates the 3 targets, no hand-engineered features needed.


In [ ]:
KNN_INTUITION_DATASET = '01_ACA_qdPCR'
KNN_INTUITION_CURVE_IDX = 0  # dataset_name[0] == 'ori_curves' -- raw curve, no preprocessing
KNN_INTUITION_SEED = 0
KNN_INTUITION_TYPICAL_FRAC = 0.4  # search the most-typical 40% of each target for the exemplar

TARGET_COLORS = {'KPC': '#1B9E77', 'NDM': '#D95F02', 'VIM': '#7570B3'}
QUERY_COLOR = '#2b2b2b'


def _pick_distinctive_curve(curves, idx_self, self_mean, other_means, typical_frac):
    """Among the most 'typical' typical_frac of idx_self (closest to their own class mean, so
    it stays a representative member of the class, not an outlier), pick the one farthest on
    average from the OTHER classes' mean curves -- a real, representative curve that is also
    maximally visually distinct from the other targets."""
    self_curves = curves[idx_self]
    d_self = np.linalg.norm(self_curves - self_mean, axis=1)
    cand = idx_self[d_self <= np.quantile(d_self, typical_frac)]
    d_other = np.mean([np.linalg.norm(curves[cand] - m, axis=1) for m in other_means], axis=0)
    return cand[np.argmax(d_other)]


def plot_knn_intuition(dataset_name=KNN_INTUITION_DATASET, curve_idx=KNN_INTUITION_CURVE_IDX,
                       seed=KNN_INTUITION_SEED, target_colors=TARGET_COLORS,
                       typical_frac=KNN_INTUITION_TYPICAL_FRAC, figsize=(7, 4.5)):
    """One real, representative-yet-distinctive curve per target (see _pick_distinctive_curve)
    plus one random held-out curve plotted as an unlabelled query ("?") -- the point being that
    classifying it is just "which coloured curve does this shape look like", no feature
    extraction required."""
    d = joblib.load(Path(config.LAB_EXP_FOLDER) / dataset_name / config.TRAINING_DATA_PATH)
    curves = np.asarray(d['dataset'][curve_idx])
    timestamps = np.asarray(d['timestamps'])
    Y_well = np.asarray(d['Y_well'])
    targets = [t for t in target_colors if t in np.unique(Y_well)]

    rng = np.random.default_rng(seed)
    query_idx = rng.integers(len(curves))
    query_curve, query_target = curves[query_idx], Y_well[query_idx]

    target_idx = {t: np.where(Y_well == t)[0] for t in targets}
    target_idx = {t: idx[idx != query_idx] for t, idx in target_idx.items()}  # never reuse the query row
    target_mean = {t: curves[idx].mean(axis=0) for t, idx in target_idx.items()}

    fig, ax = plt.subplots(figsize=figsize)
    for t in targets:
        other_means = [target_mean[o] for o in targets if o != t]
        rep_idx = _pick_distinctive_curve(curves, target_idx[t], target_mean[t], other_means, typical_frac)
        ax.plot(timestamps, curves[rep_idx], color=target_colors[t], lw=2.2, label=t)

    ax.plot(timestamps, query_curve, color=QUERY_COLOR, lw=2.2, ls='--', label='?')

    ax.set_xlabel('Cycle')
    ax.set_ylabel('Signal')
    ax.set_title(f'{dataset_name}: curve shape alone separates targets')
    ax.legend(fontsize=10, framealpha=0.9)
    fig.tight_layout()
    plt.show()
    return query_target


_knn_intuition_answer = plot_knn_intuition()
print(f'(reveal, not shown on the plot) "?" was actually: {_knn_intuition_answer}')


## CNN Intuition -- Sliding-Window Feature Extraction (01_ACA_qdPCR)

Same curve as the "?" query above. A small window (w=3) slides along it; at each position it's
reduced to one local feature (shown here as local slope -- what an edge-detecting first kernel
would naturally pick up), building up a feature-map sequence below -- illustrating what a 1D
conv layer does that raw curve-shape KNN doesn't: turn each local patch into a learned feature
before comparing/classifying.


In [ ]:
CNN_WINDOW_SIZE = 3
CNN_HIGHLIGHT_FRACS = [1/6, 1/2, 0.72]  # example window positions: flat / rising / plateau region


def plot_cnn_sliding_window_intuition(dataset_name=KNN_INTUITION_DATASET, curve_idx=KNN_INTUITION_CURVE_IDX,
                                      seed=KNN_INTUITION_SEED, window_size=CNN_WINDOW_SIZE,
                                      highlight_fracs=CNN_HIGHLIGHT_FRACS, color='#D95F02', figsize=(8, 5)):
    """Static illustration of a 1D CNN's sliding-window convolution: reuses the exact curve
    plotted as "?" in plot_knn_intuition (same dataset/curve_idx/seed) so the two figures read
    as a pair. The window's reduction to one scalar is illustrated as local slope
    (curve[end] - curve[start]) -- a stand-in for a real trained kernel's dot product, not a
    literal one, but the classic "first-layer kernels detect edges" intuition -- not the well
    id / target identity, which is irrelevant here (this figure is about the mechanism, not
    classification)."""
    d = joblib.load(Path(config.LAB_EXP_FOLDER) / dataset_name / config.TRAINING_DATA_PATH)
    curves = np.asarray(d['dataset'][curve_idx])
    timestamps = np.asarray(d['timestamps'])

    rng = np.random.default_rng(seed)
    curve = curves[rng.integers(len(curves))]

    n = len(curve)
    dt = timestamps[1] - timestamps[0]
    starts = np.arange(0, n - window_size + 1)
    feature = curve[starts + window_size - 1] - curve[starts]
    feat_x = timestamps[starts] + (window_size - 1) / 2 * dt
    highlight_starts = [int(f * (n - window_size)) for f in highlight_fracs]
    alphas = np.linspace(0.28, 0.9, len(highlight_starts))

    fig, (ax_curve, ax_feat) = plt.subplots(
        2, 1, figsize=figsize, sharex=True, height_ratios=[3, 1],
        gridspec_kw={'hspace': 0.08})

    ax_curve.plot(timestamps, curve, color='#1f77b4', lw=2.2, zorder=3)
    ax_curve.set_ylabel('Signal')
    ax_curve.set_title(f'{dataset_name}: 1D CNN sliding window (w={window_size}) -- '
                       f'each local patch \u2192 one feature value', fontsize=11)
    ax_curve.annotate('slides \u2192', xy=(timestamps[highlight_starts[-1]] + 2 * dt, ax_curve.get_ylim()[1] * 0.15),
                      fontsize=10, color=color, fontweight='bold')

    ax_feat.plot(feat_x, feature, color='#888888', lw=1.1, zorder=2)
    ax_feat.axhline(0, color='#cccccc', lw=0.8, zorder=1)
    ax_feat.set_ylabel('Local\nslope', fontsize=9)
    ax_feat.set_xlabel('Cycle')

    for a, s in zip(alphas, highlight_starts):
        x0, x1 = timestamps[s], timestamps[s + window_size - 1]
        ax_curve.axvspan(x0, x1, color=color, alpha=a, lw=0, zorder=2)

        fx, fy = feat_x[s], feature[s]
        ax_feat.scatter([fx], [fy], color=color, alpha=a, s=70, zorder=5, edgecolor='black', linewidth=0.6)
        con = ConnectionPatch(xyA=((x0 + x1) / 2, curve[s:s + window_size].min()), coordsA=ax_curve.transData,
                              xyB=(fx, fy), coordsB=ax_feat.transData,
                              color=color, alpha=a, lw=1.2, ls='--', zorder=1)
        fig.add_artist(con)

    fig.tight_layout()
    plt.show()


plot_cnn_sliding_window_intuition()


## GRU Intuition -- Sequential Memory (01_ACA_qdPCR)

Same curve again. Unlike the CNN's small fixed window, a GRU reads the curve one step at a
time and carries a *memory* forward step-to-step ($h_{t-1} \to h_t$) that can, in principle,
span the whole curve seen so far -- not just a local patch. Shading grows to show "everything
read so far" at each step; the strip below is an illustrative memory trace.


In [ ]:
GRU_EMA_ALPHA = 0.3  # illustrative memory blend rate -- NOT a trained GRU's gate weights
GRU_HIGHLIGHT_FRACS = [0.30, 0.55, 0.80]


def plot_gru_memory_intuition(dataset_name=KNN_INTUITION_DATASET, curve_idx=KNN_INTUITION_CURVE_IDX,
                              seed=KNN_INTUITION_SEED, ema_alpha=GRU_EMA_ALPHA,
                              highlight_fracs=GRU_HIGHLIGHT_FRACS, color='#66A61E', figsize=(8, 5.2)):
    """Static illustration of a GRU's sequential, memory-carrying processing -- reuses the same
    curve as plot_knn_intuition/plot_cnn_sliding_window_intuition so all three figures read as
    one set. 'Memory' is an exponential moving average -- a simplified stand-in for a trained
    GRU's gated hidden-state update (same illustrative-not-literal disclaimer as the CNN
    figure's local-slope feature): h[0] = curve[0], h[t] = alpha*curve[t] + (1-alpha)*h[t-1].
    The key visual contrast with the CNN figure: the highlighted region GROWS from the start of
    the curve at each step (unfolding history), not a small fixed-size local window."""
    d = joblib.load(Path(config.LAB_EXP_FOLDER) / dataset_name / config.TRAINING_DATA_PATH)
    curves = np.asarray(d['dataset'][curve_idx])
    timestamps = np.asarray(d['timestamps'])

    rng = np.random.default_rng(seed)
    curve = curves[rng.integers(len(curves))]
    n = len(curve)

    h = np.empty(n)
    h[0] = curve[0]
    for t in range(1, n):
        h[t] = ema_alpha * curve[t] + (1 - ema_alpha) * h[t - 1]

    highlight_idx = [int(f * (n - 1)) for f in highlight_fracs]
    alphas = np.linspace(0.28, 0.9, len(highlight_idx))

    fig, (ax_curve, ax_h) = plt.subplots(
        2, 1, figsize=figsize, sharex=True, height_ratios=[3, 1],
        gridspec_kw={'hspace': 0.08})

    ax_curve.plot(timestamps, curve, color='#1f77b4', lw=2.2, zorder=3)
    ax_curve.set_ylabel('Signal')
    ax_curve.set_title(f'{dataset_name}: GRU reads the curve step-by-step, carrying a memory forward',
                       fontsize=11)

    seg_starts = [0] + highlight_idx[:-1]
    for a, s_idx, idx in zip(alphas, seg_starts, highlight_idx):
        ax_curve.axvspan(timestamps[s_idx], timestamps[idx], color=color, alpha=a * 0.4, lw=0, zorder=1)
    for a, idx in zip(alphas, highlight_idx):
        ax_curve.plot([timestamps[idx]], [curve[idx]], 'o', color=color, alpha=a, ms=8,
                     markeredgecolor='black', markeredgewidth=0.6, zorder=4)

    # h_{t-1} -> h_t chain, drawn as curved arrows above the curve between consecutive markers.
    y_top = ax_curve.get_ylim()[1]
    for i in range(len(highlight_idx) - 1):
        x0, x1 = timestamps[highlight_idx[i]], timestamps[highlight_idx[i + 1]]
        arrow = FancyArrowPatch((x0, y_top * 1.05), (x1, y_top * 1.05),
                                connectionstyle="arc3,rad=-0.3", arrowstyle='-|>',
                                mutation_scale=14, color=color, lw=1.4, clip_on=False, zorder=5)
        ax_curve.add_patch(arrow)
    for i, idx in enumerate(highlight_idx):
        ax_curve.annotate(f'$h_{i+1}$', xy=(timestamps[idx], y_top * 1.12), color=color,
                          fontsize=11, ha='center', fontweight='bold', annotation_clip=False)
    ax_curve.set_ylim(top=y_top * 1.25)

    ax_h.plot(timestamps, h, color='#888888', lw=1.3, zorder=2)
    ax_h.set_ylabel('Memory\n(EMA)', fontsize=9)
    ax_h.set_xlabel('Cycle')

    for a, idx in zip(alphas, highlight_idx):
        ax_h.scatter([timestamps[idx]], [h[idx]], color=color, alpha=a, s=70, zorder=5,
                    edgecolor='black', linewidth=0.6)
        con = ConnectionPatch(xyA=(timestamps[idx], curve[idx]), coordsA=ax_curve.transData,
                              xyB=(timestamps[idx], h[idx]), coordsB=ax_h.transData,
                              color=color, alpha=a, lw=1.2, ls='--', zorder=1)
        fig.add_artist(con)

    fig.tight_layout()
    plt.show()


plot_gru_memory_intuition()
